# Embeddings and Recommendations
- I wish I learnt this before
- This is so powerful
### Most optimal num of dimension for embeddings
- $$embedding\_dim \approx \min(50, \frac{\text{\#users + \#items}}{2})$$
- Netflix and other recommender system papers often use 50–200 dimensions for large datasets.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

/Users/shubhambhardwaj/Shubham/datascience/study/LLM/embeddings/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [39]:
def get_embeddings(num_users,num_movies, embedding_dim):
    # Embeddings for users and movies
    user_embeddings = nn.Embedding(num_users, embedding_dim)
    movie_embeddings = nn.Embedding(num_movies, embedding_dim)

    # Initialize embeddings (optional, PyTorch does it randomly)
    nn.init.normal_(user_embeddings.weight, mean=0, std=0.1)
    nn.init.normal_(movie_embeddings.weight, mean=0, std=0.1)

    # Define optimizer
    optimizer = optim.SGD(list(user_embeddings.parameters()) + list(movie_embeddings.parameters()), lr=0.1)

    # Training loop
    for epoch in range(1000):
        optimizer.zero_grad()
        
        # Compute predictions: dot product of user and movie embeddings
        preds = torch.matmul(user_embeddings.weight, movie_embeddings.weight.t())
        
        # Compute loss (MSE)
        loss = ((preds - ratings) ** 2).mean()
        loss.backward()
        optimizer.step()

    # Final embeddings
    print("User embeddings:\n", user_embeddings.weight)
    print("Movie embeddings:\n", movie_embeddings.weight)

    # Predicted ratings
    preds = torch.matmul(user_embeddings.weight, movie_embeddings.weight.t())
    print(f"user weights {user_embeddings.weight}")
    print("Predicted ratings:\n", preds)
    pred_probs = torch.sigmoid(preds)
    print("Predicted probabilities:\n", pred_probs)
    print("Intial data")
    print(ratings)

### Effect of embedding dims on the model

In [40]:
# Toy dataset: 3 users, 4 movies
# 1 if user likes movie, 0 otherwise
ratings = torch.tensor([
    [1, 0, 1, 0],  # User 1
    [0, 1, 0, 1],  # User 2
    [1, 1, 0, 0],  # User 3
], dtype=torch.float32)

num_users, num_movies = ratings.shape
embedding_dim = 2  # small dimension to trace easily
get_embeddings(num_users=num_users, num_movies=num_movies, embedding_dim=2)

User embeddings:
 Parameter containing:
tensor([[ 1.0520,  0.2492],
        [-0.4593,  0.9828],
        [ 0.4192,  0.8711]], requires_grad=True)
Movie embeddings:
 Parameter containing:
tensor([[ 0.9220,  0.5362],
        [-0.1508,  1.0524],
        [ 0.6960,  0.0709],
        [-0.3767,  0.5872]], requires_grad=True)
user weights Parameter containing:
tensor([[ 1.0520,  0.2492],
        [-0.4593,  0.9828],
        [ 0.4192,  0.8711]], requires_grad=True)
Predicted ratings:
 tensor([[ 1.1035,  0.1036,  0.7499, -0.2500],
        [ 0.1035,  1.1036, -0.2500,  0.7501],
        [ 0.8536,  0.8535,  0.3536,  0.3535]], grad_fn=<MmBackward0>)
Predicted probabilities:
 tensor([[0.7509, 0.5259, 0.6792, 0.4378],
        [0.5259, 0.7509, 0.4378, 0.6792],
        [0.7013, 0.7013, 0.5875, 0.5875]], grad_fn=<SigmoidBackward0>)
Intial data
tensor([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 1., 0., 0.]])


In [ ]:
get_embeddings(num_users=num_users, num_movies=num_movies, embedding_dim=(num_users + num_movies) // 2)

### Real world example simulation with movie and user features
- Users have the following features:
1. genre they like for eg: action, comedy, drama - Categorical
- Movies have the following features:
1. genre - Categorical
2. popularity - float

We will one-hot encode the categorical features

The Target data called rating is a mapping of users to movies
- row:  user
- col: movie
There is a 1 if the user likes the movie and a 0 if the user doesnot like the movie and a nan if they dont have any preferences on the movie.  

### Flow:
- #### Training
``` user_features ---> [User Tower NN] ---> u_proj (requires_grad=True)
                                                |
                                                v
movie_features ---> [Movie Tower NN] ---> m_proj (requires_grad=True)
                                                |
                                                v
                                    preds = u_proj @ m_proj.T
                                                |
                                                v
                                            loss (requires_grad=True)
                                                |
                                                v
                                         loss.backward()
                                                |
                          -------------------- gradients flow --------------------
                          |                                                       |
                  User Tower weights                                      Movie Tower weights
                (updated by optimizer)                                   (updated by optimizer)
```
- #### Inference
```
with torch.no_grad():
    user_features ---> [User Tower NN] ---> u_proj (requires_grad=False)
                                                  |
                                                  v
    movie_features ---> [Movie Tower NN] ---> m_proj (requires_grad=False)
                                                  |
                                                  v
                                      preds = u_proj @ m_proj.T
```

In [ ]:
# === 1. Toy ratings matrix (4 users × 5 movies) ===
ratings = torch.tensor([
    [1, 0, 1, float('nan'), 0],
    [0, 1, float('nan'), 1, 0],
    [1, float('nan'), 0, 1, 1],
    [float('nan'), 1, 0, 0, 1],
], dtype=torch.float32)

num_users, num_movies = ratings.shape
embedding_dim = 3  # latent dimension (for learned embeddings)

# === 2. Movie features (Action, Comedy, Drama, Popularity) ===
movie_features = torch.tensor([
    [1, 0, 0, 0.8],
    [0, 1, 0, 0.6],
    [0, 0, 1, 0.9],
    [1, 0, 0, 0.3],
    [0, 1, 0, 0.5],
], dtype=torch.float32)

# === 3. User features (likes_action, likes_comedy, likes_drama) ===
user_features = torch.tensor([
    [1, 0, 1],
    [0, 1, 0],
    [1, 1, 0],
    [0, 1, 1],
], dtype=torch.float32)

# === 4. Embedding layers ===
user_embeddings = nn.Embedding(num_users, embedding_dim)
movie_embeddings = nn.Embedding(num_movies, embedding_dim)
nn.init.normal_(user_embeddings.weight, mean=0, std=0.1)
nn.init.normal_(movie_embeddings.weight, mean=0, std=0.1)

# === 5. Projection layers (map user/movie to same 8D latent space) ===
latent_dim = 8
user_proj = nn.Linear(embedding_dim + user_features.shape[1], latent_dim)
movie_proj = nn.Linear(embedding_dim + movie_features.shape[1], latent_dim)

# === 6. Optimizer ===
optimizer = optim.Adam(
    list(user_embeddings.parameters()) +
    list(movie_embeddings.parameters()) +
    list(user_proj.parameters()) +
    list(movie_proj.parameters()),
    lr=0.01
)

# === 7. Mask for known ratings ===
mask = ~torch.isnan(ratings)

# === 8. Training loop ===
for epoch in range(300):
    optimizer.zero_grad()
    
    # Get embeddings
    u_emb = user_embeddings.weight
    m_emb = movie_embeddings.weight
    
    # Combine embeddings with side features
    u_input = torch.cat([u_emb, user_features], dim=1)
    m_input = torch.cat([m_emb, movie_features], dim=1)
    
    # Project both to common latent space
    u_proj = user_proj(u_input)
    m_proj = movie_proj(m_input)
    
    # Predicted ratings = dot product of user/movie latent vectors
    preds = torch.matmul(u_proj, m_proj.t())
    
    # Masked MSE loss (only compute where ratings are known)
    loss = ((preds - torch.nan_to_num(ratings)) ** 2 * mask).sum() / mask.sum()
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/1000, Loss = {loss.item():.4f}")

# === 9. Inspect results ===
print("\nFinal User Embeddings:\n", user_embeddings.weight)
print("\nFinal Movie Embeddings:\n", movie_embeddings.weight)

# Predictions
with torch.no_grad():
    u_proj = user_proj(torch.cat([user_embeddings.weight, user_features], dim=1))
    m_proj = movie_proj(torch.cat([movie_embeddings.weight, movie_features], dim=1))
    preds = torch.matmul(u_proj, m_proj.t())
    print("\nPredicted Ratings:\n", preds)

    # New movie (e.g. "Sci-Fi Action, popularity=0.7")
    new_movie_features = torch.tensor([[1, 0, 1, 0.7]])  # (Action+Drama)
    new_movie_emb = torch.zeros((1, embedding_dim))  # no learned embedding
    new_movie_input = torch.cat([new_movie_emb, new_movie_features], dim=1)
    new_movie_latent = movie_proj(new_movie_input)
    
    # Get all existing users' latent vectors
    u_proj = user_proj(torch.cat([user_embeddings.weight, user_features], dim=1))
    
    # Predict how much each user would like this new movie
    new_movie_scores = torch.matmul(u_proj, new_movie_latent.t())
    print("\nPredicted preference for NEW movie:\n", new_movie_scores.squeeze())
    
    # New user (likes action and comedy)
    new_user_features = torch.tensor([[1, 1, 0]])
    new_user_emb = torch.zeros((1, embedding_dim))  # cold-start user
    new_user_input = torch.cat([new_user_emb, new_user_features], dim=1)
    new_user_latent = user_proj(new_user_input)
    
    # Get all existing movies' latent vectors
    m_proj = movie_proj(torch.cat([movie_embeddings.weight, movie_features], dim=1))
    
    # Predict which movies this user might like
    new_user_scores = torch.matmul(new_user_latent, m_proj.t())
    print("\nPredicted preferences for NEW user across movies:\n", new_user_scores.squeeze())

Epoch 200/1000, Loss = 0.0000

Final User Embeddings:
 Parameter containing:
tensor([[-0.2988,  0.3641, -0.2314],
        [-0.4191,  0.0352, -0.3293],
        [ 0.0704, -0.0315,  0.3108],
        [ 0.4221, -0.2438,  0.0304]], requires_grad=True)

Final Movie Embeddings:
 Parameter containing:
tensor([[-0.4500,  0.0678,  0.1331],
        [ 0.0998,  0.1808, -0.2945],
        [-0.2347, -0.1612,  0.0710],
        [ 0.0551, -0.3562, -0.2905],
        [-0.2020,  0.0868,  0.3813]], requires_grad=True)

Predicted Ratings:
 tensor([[ 1.0000e+00,  2.0862e-07,  1.0000e+00,  1.2692e+00, -1.4781e-07],
        [ 2.0862e-07,  1.0000e+00, -2.0718e-01,  1.0000e+00,  1.3632e-07],
        [ 1.0000e+00,  1.1612e+00, -1.7881e-07,  1.0000e+00,  1.0000e+00],
        [ 1.9100e-01,  1.0000e+00,  2.8312e-07, -3.5763e-07,  1.0000e+00]])
